# Checkpoint 2 — hosted GPU experiment

Runs the same CLI used locally: verified public fixed references → Gaussian preparation → training-only pilot → frozen nine-run matrix → held-out benchmark. Select a T4 GPU and **Runtime version 2026.07 (Python 3.12)** before setup. The latest Python 3.13 runtime is incompatible with these dependency pins. This notebook does not create the report, slides, video, or submission ZIP.

The public references are **not demonstrated noise-free anatomy**. Measurements concern recovery after *additional* Gaussian corruption. No clinical claim. Download public data directly; do not publish volumes under the code license.

Before uploading, export the current implementation locally:
```sh
python NGPS/scripts/export_checkpoint2_source.py --output /tmp/checkpoint2-source.zip
```
Upload that ZIP in the next cell, or supply its path on Kaggle. This avoids accidentally running the older code currently on GitHub.


In [ ]:
from pathlib import Path
import os, sys, subprocess, zipfile
if sys.version_info[:2] not in ((3, 11), (3, 12)):
    raise RuntimeError('Select Runtime > Change runtime type > Runtime version 2026.07 (Python 3.12), then reconnect before setup.')
def run_logged(command, check=True, cwd=None):
    print('Running:', ' '.join(map(str, command)), flush=True)
    process = subprocess.Popen(command, cwd=cwd, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line, end='', flush=True)
    result = subprocess.CompletedProcess(command, process.wait())
    if check and result.returncode:
        raise subprocess.CalledProcessError(result.returncode, command)
    return result
SOURCE_ZIP = None  # Kaggle example: '/kaggle/input/project-ml-source/checkpoint2-source.zip'
if SOURCE_ZIP is None:
    from google.colab import files
    uploaded = files.upload()
    archives = [name for name in uploaded if name.endswith('.zip')]
    if len(archives) != 1:
        raise ValueError('Upload exactly one checkpoint2 source ZIP')
    SOURCE_ZIP = archives[0]
CODE_ROOT = Path('/content/project_ml' if Path('/content').exists() else '/kaggle/working/project_ml')
NGPS = CODE_ROOT / 'NGPS'
NGPS.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(SOURCE_ZIP) as archive:
    for member in archive.infolist():
        if NGPS.resolve() not in (NGPS / member.filename).resolve().parents:
            raise ValueError('Unsafe archive member')
    archive.extractall(NGPS)
run_logged([sys.executable, '-m', 'pip', 'install', '-r', str(NGPS / 'requirements-colab.txt')], check=True)
print('If torch was previously imported, restart the runtime and rerun before continuing.')


In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime; medical production runs require CUDA')
print(torch.__version__, torch.cuda.get_device_name(0))
if Path('/content').exists():
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = Path('/content/drive/MyDrive/ProjectML-Checkpoint2')
else:
    # Kaggle session disk is ephemeral: save a notebook version with outputs after training.
    WORK = Path('/kaggle/working/ProjectML-Checkpoint2')
WORK.mkdir(parents=True, exist_ok=True)
assert not str(WORK).startswith('/content/project_ml')
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
def cli(script, *args):
    run_logged([sys.executable, str(NGPS / 'scripts' / script), *map(str, args)], check=True, cwd=CODE_ROOT)
cli('smoke_test.py')
run_logged([sys.executable, '-m', 'unittest', 'discover', '-s', str(NGPS / 'tests'), '-v'], check=True, cwd=CODE_ROOT)
SYNTHETIC = WORK / 'synthetic-check'
if not SYNTHETIC.exists():
    cli('synthetic_pipeline.py', '--work-dir', SYNTHETIC, '--checkpoint2')


## Obtain public data and prepare observations
The script downloads all ten authoritative MINC files, audits their headers and conversions, and writes a receipt. No source volumes are uploaded to a third-party model service. Internet access is required (enable it in Kaggle). Existing outputs are preserved; failed partial acquisition should use a new directory.


In [ ]:
SOURCE = WORK / 'sources' / 'subjects.json'
RECEIPT = WORK / 'brainweb_verification.json'
PREPARED = WORK / 'prepared'
if not RECEIPT.exists():
    cli('verify_brainweb.py', '--acquire', '--manifest', SOURCE,
        '--output', RECEIPT, '--cache', WORK / 'download-cache', '--accept-fixed-reference')
if not (PREPARED / 'manifest.json').exists():
    cli('checkpoint2.py', 'prepare', '--manifest', SOURCE, '--verification', RECEIPT, '--output-dir', PREPARED)
MANIFEST = PREPARED / 'manifest.json'
CONFIG = NGPS / 'configs/checkpoint2.yaml'


## Pilot and freeze before testing
The pilot uses training subject 04, sigma 0.06, 16 center slices, one epoch, and batch 1 for each objective. It estimates a common workload with a 1.5 safety factor. If validation alone exceeds the cap, lower the common epoch count in a new config and rerun the pilot, or predeclare a larger cap with a written reason. Never alter a frozen protocol or choose settings from clean validation/test scores.


In [ ]:
PILOT = WORK / 'pilot-v1'
EXPERIMENT = WORK / 'experiment-v1'
if not (PILOT / 'pilot.json').exists():
    cli('checkpoint2.py', 'pilot', '--manifest', MANIFEST, '--config', CONFIG, '--output-dir', PILOT, '--device', 'cuda')
if not (EXPERIMENT / 'protocol.json').exists():
    cli('checkpoint2.py', 'freeze', '--manifest', MANIFEST, '--config', CONFIG,
        '--pilot', PILOT / 'pilot.json', '--output-dir', EXPERIMENT)
print((EXPERIMENT / 'protocol.json').read_text())


In [ ]:
PROTOCOL = EXPERIMENT / 'protocol.json'
cli('checkpoint2.py', 'train', '--protocol', PROTOCOL, '--device', 'cuda')
# Completed matching runs are skipped on rerun. Partial runs are preserved and
# rejected: use a separately named experiment to rerun, never delete evidence.
for sigma in ('0.03', '0.06', '0.10'):
    diagnostic = EXPERIMENT / f'diagnostics_s{sigma}.npz'
    if not diagnostic.exists():
        cli('checkpoint2.py', 'diagnostics', '--manifest', MANIFEST,
            '--config', EXPERIMENT / 'training.yaml', '--output', diagnostic,
            '--subject', 'subject04', '--sigma', sigma,
            '--checkpoint', EXPERIMENT / 'runs' / f'ngps_s{sigma}' / 'model_final.pth')


## Final held-out benchmark
All nine selected checkpoints must pass preflight. This opens held-out references and records testing as started. Keep failures and unmet targets. Repeated benchmark attempts must not become tuning. Success is zero PSNR deficit against the stronger trained comparator at every sigma.


In [ ]:
if not (EXPERIMENT / 'results' / 'COMPLETE.json').exists():
    cli('checkpoint2.py', 'benchmark', '--protocol', PROTOCOL, '--device', 'cuda')
print((EXPERIMENT / 'results' / 'success.csv').read_text())
from IPython.display import display, Image
for figure in sorted((EXPERIMENT / 'results' / 'figures').glob('*.png')):
    display(Image(filename=str(figure)))


In [ ]:
# Reports, predictions, trained checkpoints, per-run source snapshots, diagnostics,
# split/noise manifests, and source receipts remain in WORK. Exclude public raw and
# prepared volumes from the export; obtain them directly using the documented CLI.
# Colab uses mounted Drive throughout. Kaggle: Save Version with outputs before exit.
import json
print('Persistent artifact root:', WORK)
print('Completion:', json.loads((EXPERIMENT / 'results/COMPLETE.json').read_text()))
